# Fine-tune PP-OCRv6 medium on a mix of writers (Colab) - v7

Same recipe as the v6 notebook (same two PaddleOCR patches, same low learning rate, CTC head only),
but the training data is no longer one hand. What changed:

1. **Mixed training data.** Your journal crops, augmented copies of them, optional pages from other writers
   (`other_writers/` in the repo) and an IAM line subset, in a fixed ratio (`OWN_FRACTION`).
2. **Checkpoint chosen on other writers too.** The validation set that picks `best_accuracy` combines your
   own val lines, IAM lines and (if present) your collected other-writer val lines. Before, it was only your own handwriting.
3. **Mild augmentation, pre-generated** (small rotation, contrast, blur, ruled lines), so RecAug stays off.
4. **Fewer epochs** (15 instead of 30). Your old validation curve was flat after about epoch 12.
5. **Small backup.** Only `best_accuracy.pdparams` (about 230 MB) is downloaded, not a 1.7 GB zip.
6. **A results table** that scores stock and the new model on each data source separately, so you can see
   whether the model got better on other writers without getting worse on yours.

## Before you run
- `journal_crops/` (with `train_list.txt`, `val_list.txt`, `test_list.txt`) must be on GitHub. All crops should be made the same way (re-exported with `reexport_crops_polygon.py`).
- IAM subset (recommended): run `ocr_training/prepare_iam_data.py` and `ocr_training/make_iam_subset.py` on your PC (commands in step 5) to get `iam_subset.zip`. If you skip it, the notebook still runs, but only on your own data.
- Other writers (optional, add later): push `other_writers/train_list.txt` and `other_writers/val_list.txt` with crops to GitHub. Without them, nothing in this run is validated on a writer other than you or IAM.

Do not use Runtime > Run all. Step 5 stops to ask for a file upload. Run the cells top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi | head -15

## 2. Settings

In [ ]:
import os

REPO_URL = "https://github.com/chuajeromepython/CS-PADDLE-OCR.git"
REPO_DIR = "/content/CS-PADDLE-OCR"
DATA_DIR = REPO_DIR + "/"

# your own journal crops (paths inside these lists are relative to the repo root)
OWN_TRAIN = REPO_DIR + "/journal_crops/train_list.txt"
OWN_VAL   = REPO_DIR + "/journal_crops/val_list.txt"
OWN_TEST  = REPO_DIR + "/journal_crops/test_list.txt"

# crops from other writers you collect later (optional, same list format)
OTHER_DIR   = REPO_DIR + "/other_writers"
OTHER_TRAIN = OTHER_DIR + "/train_list.txt"
OTHER_VAL   = OTHER_DIR + "/val_list.txt"

# IAM subset, unpacked here in step 5
IAM_DIR = "/content/iam_data"

# ---- data mix ----
OWN_FRACTION     = 0.4    # target share of training lines that are YOUR handwriting (real + augmented)
AUG_COPIES_OWN   = 3      # augmented copies made per crop of yours (3 keeps your hand about a third of the mix)
AUG_COPIES_OTHER = 1      # augmented copies per crop from other writers (SMHD is already large)
IAM_VAL_MAX      = 150    # max IAM lines used for validation
IAM_VAL_FORMS    = 0.15   # share of IAM forms held out for validation
SEED = 42

# ---- training ----
EPOCHS = 15
BATCH  = 16
LR_STR = "0.00002"        # written out in full: YAML reads 2e-05 as text, not a number
IMG_W  = 704              # width every crop is resized/padded to (13:1 lines at height 48)

PADDLEOCR_DIR = "/content/PaddleOCR"
PRETRAIN_DIR  = PADDLEOCR_DIR + "/pretrain"
PRETRAINED    = PRETRAIN_DIR + "/PP-OCRv6_medium_rec_pretrained"   # PaddleOCR appends .pdparams
OUTPUT_DIR    = "/content/output/journal_v7"
CONFIG_PATH   = PADDLEOCR_DIR + "/configs/rec/PP-OCRv6/PP-OCRv6_medium_rec_journal_v7.yml"
WORK          = "/content/work"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(WORK, exist_ok=True)

In [ ]:
%cd /content
WHEEL = "https://paddle-whl.cdn.bcebos.com/stable/cu126/paddlepaddle-gpu/paddlepaddle_gpu-3.3.1-cp313-cp313-linux_x86_64.whl"
!wget -c --tries=0 --timeout=60 --waitretry=5 -nv -O /content/paddlepaddle_gpu-3.3.1-cp313-cp313-linux_x86_64.whl {WHEEL}
!pip install -q /content/paddlepaddle_gpu-3.3.1-cp313-cp313-linux_x86_64.whl
import paddle
print(paddle.__version__, "CUDA build:", paddle.is_compiled_with_cuda())

## 3. Install PaddlePaddle and clone PaddleOCR

In [ ]:
!pip install -q paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
import os
%cd /content
if not os.path.isdir(PADDLEOCR_DIR + "/.git"):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd /content/PaddleOCR
!pip install -q -r requirements.txt
!pip install -q rapidfuzz
import paddle
print(paddle.__version__, "CUDA build:", paddle.is_compiled_with_cuda())

## 4. Get your data from GitHub

In [ ]:
%cd /content
if not os.path.isdir(REPO_DIR + "/.git"):
    !git clone --depth 1 {REPO_URL}
else:
    !cd {REPO_DIR} && git pull -q
for p in (OWN_TRAIN, OWN_VAL, OWN_TEST, OTHER_TRAIN, OTHER_VAL):
    if os.path.exists(p):
        print(f"{os.path.relpath(p, REPO_DIR):32} {sum(1 for l in open(p, encoding='utf-8') if l.strip()):5d} lines")
    else:
        print(f"{os.path.relpath(p, REPO_DIR):32} (not found)")

## 5. IAM subset (other writers, recommended)

Your old IAM model used IAM *word* crops. This notebook needs IAM *line* crops, so make a new small subset on your PC
(about 2500 lines, about 60-100 MB, small enough to upload in one go):

```
# download ascii/lines.txt and data/lines.tgz from the IAM site (same account as words.zip) and extract lines.tgz
python ocr_training/prepare_iam_data.py --lines-txt C:\IAM\ascii\lines.txt --format line --img-root C:\IAM\lines --out-dir iam_lines_labels --val-split 0.0
python ocr_training/make_iam_subset.py --lists iam_lines_labels\train_list.txt --n 2500 --out iam_subset
```

That creates `iam_subset.zip`. Run the cell below and choose that file. To skip IAM, close the file dialog without choosing anything.

In [ ]:
import zipfile

if not os.path.exists(IAM_DIR + "/iam_labels.txt"):
    from google.colab import files
    try:
        up = files.upload()
    except Exception as e:
        up = {}
        print("No file chosen:", e)
    for name in up:
        if name.lower().endswith(".zip"):
            try:
                os.makedirs(IAM_DIR, exist_ok=True)
                with zipfile.ZipFile(name) as z:
                    z.extractall(IAM_DIR)
            except zipfile.BadZipFile:
                print(f"{name} is not a complete zip (upload was cut off). Delete it and upload again.")

IAM_OK = os.path.exists(IAM_DIR + "/iam_labels.txt")
print("IAM subset available:", IAM_OK)
if not IAM_OK:
    print("WARNING: training without IAM. Only your own handwriting (and collected other writers, if any) will be used.")

## 6. Load and sanity-check the labels

Drops any line whose image is missing, whose text has a character the v6 dictionary does not contain, or that is longer than 100 characters, and prints how many were dropped from each source.

In [ ]:
import random

dict_path = PADDLEOCR_DIR + "/ppocr/utils/dict/ppocrv6_dict.txt"
valid = set(open(dict_path, encoding="utf-8").read().split("\n")) | {" "}

def read_list(path, root, max_len=100):
    """Returns ([(absolute_image_path, text), ...], {reason: count})."""
    rows, skipped = [], {"missing image": 0, "bad character": 0, "too long": 0, "empty text": 0}
    for ln in open(path, encoding="utf-8"):
        ln = ln.rstrip("\n")
        if not ln or "\t" not in ln:
            continue
        p, text = ln.split("\t", 1)
        full = p if os.path.isabs(p) else os.path.join(root, p)
        if not text.strip():
            skipped["empty text"] += 1
        elif not os.path.exists(full):
            skipped["missing image"] += 1
        elif any(c not in valid for c in text):
            skipped["bad character"] += 1
        elif len(text) > max_len:
            skipped["too long"] += 1
        else:
            rows.append((full, text))
    return rows, skipped

def load(name, path, root):
    if not os.path.exists(path):
        return []
    rows, sk = read_list(path, root)
    dropped = {k: v for k, v in sk.items() if v}
    print(f"{name:22} {len(rows):5d} lines" + (f"   dropped: {dropped}" if dropped else ""))
    return rows

own_train   = load("own train", OWN_TRAIN, REPO_DIR)
own_val     = load("own val",   OWN_VAL,   REPO_DIR)
own_test    = load("own test",  OWN_TEST,  REPO_DIR)
other_train = load("other writers train", OTHER_TRAIN, REPO_DIR)
other_val   = load("other writers val",   OTHER_VAL,   REPO_DIR)
iam_all     = load("IAM subset", IAM_DIR + "/iam_labels.txt", IAM_DIR) if IAM_OK else []

assert own_train and own_val, "journal_crops lists are missing or empty. Check step 4 (folder must be named journal_crops)."
HAVE_OTHER = bool(other_train and other_val)
if other_train and not other_val:
    print("NOTE: other_writers/train_list.txt exists but val_list.txt is empty/missing. Other writers are used for training only.")

## 7. Build the training mix and the validation sets

- **Augmentation** is pre-generated into `/content/work/aug_*` (small rotation, brightness/contrast, blur, ruled lines drawn behind the ink).
- **IAM is split by form**, not by line, so the IAM validation lines come from pages the model never trained on.
- **Training mix:** all your real lines + augmented copies + other writers (all of it, if present) + enough IAM lines that your own handwriting is about `OWN_FRACTION` of the total.
- **Checkpoint selection set (`VAL_MIX`):** your own val + IAM val + other-writer val. The best epoch is picked on this, so a model that only improves on your handwriting will not win.

In [ ]:
import cv2, numpy as np

rng = random.Random(SEED)

def augment(img, rng):
    h, w = img.shape[:2]
    out = img.copy()
    # small rotation
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-1.5, 1.5), 1.0)
    out = cv2.warpAffine(out, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    # brightness / contrast
    out = np.clip(out.astype(np.float32) * rng.uniform(0.75, 1.25) + rng.uniform(-20, 20), 0, 255).astype(np.uint8)
    # blur
    if rng.random() < 0.4:
        k = rng.choice([3, 5])
        out = cv2.GaussianBlur(out, (k, k), 0)
    # ruled paper line behind the ink (darken only where the page is light)
    if rng.random() < 0.5:
        layer = np.full_like(out, 255)
        y0 = rng.randint(int(h * 0.55), h - 1)
        y1 = int(np.clip(y0 + rng.uniform(-0.02, 0.02) * w, 0, h - 1))
        color = rng.choice([(200, 150, 120), (180, 180, 180), (150, 170, 200)])
        cv2.line(layer, (0, y0), (w - 1, y1), color, rng.choice([1, 1, 2]))
        out = np.minimum(out, layer)
    return out

def make_aug(rows, copies, tag):
    d = f"{WORK}/aug_{tag}"
    os.makedirs(d, exist_ok=True)
    out = []
    for path, text in rows:
        img = cv2.imread(path)
        if img is None:
            continue
        stem = os.path.splitext(os.path.basename(path))[0]
        for k in range(copies):
            p = f"{d}/{stem}_aug{k}.png"
            cv2.imwrite(p, augment(img, rng))
            out.append((p, text))
    return out

own_aug   = make_aug(own_train, AUG_COPIES_OWN, "own")
other_aug = make_aug(other_train, AUG_COPIES_OTHER, "other")

# ---- IAM split by form
def form_id(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    return "-".join(stem.split("-")[:2])

iam_train, iam_val = [], []
if iam_all:
    forms = sorted({form_id(p) for p, _ in iam_all})
    rng.shuffle(forms)
    val_forms = set(forms[: max(1, round(len(forms) * IAM_VAL_FORMS))])
    iam_val_all = [r for r in iam_all if form_id(r[0]) in val_forms]
    iam_train_all = [r for r in iam_all if form_id(r[0]) not in val_forms]
    rng.shuffle(iam_val_all)
    iam_val = iam_val_all[:IAM_VAL_MAX]

n_own = len(own_train) + len(own_aug)
n_other = len(other_train) + len(other_aug)
if iam_all:
    want_iam = max(0, round(n_own / OWN_FRACTION) - n_own - n_other)
    rng.shuffle(iam_train_all)
    iam_train = iam_train_all[:want_iam]

train_rows = own_train + own_aug + other_train + other_aug + iam_train
rng.shuffle(train_rows)

def write_list(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for p, t in rows:
            f.write(f"{p}\t{t}\n")

TRAIN_MIX = WORK + "/train_mix.txt"
VAL_OWN   = WORK + "/val_own.txt"
TEST_OWN  = WORK + "/test_own.txt"
VAL_IAM   = WORK + "/val_iam.txt"
VAL_OTHER = WORK + "/val_other.txt"
VAL_MIX   = WORK + "/val_mix.txt"

write_list(TRAIN_MIX, train_rows)
write_list(VAL_OWN, own_val)
write_list(TEST_OWN, own_test)
write_list(VAL_IAM, iam_val)
write_list(VAL_OTHER, other_val)
write_list(VAL_MIX, own_val + iam_val + other_val)

n_train = len(train_rows)
steps_per_epoch = max(n_train // BATCH, 1)
share_own = n_own / n_train

print(f"TRAIN MIX: {n_train} lines  ({steps_per_epoch} steps per epoch)")
print(f"  your handwriting: {len(own_train)} real + {len(own_aug)} augmented = {n_own}  ({share_own:.0%})")
print(f"  collected others: {len(other_train)} real + {len(other_aug)} augmented = {n_other}")
print(f"  IAM:              {len(iam_train)}")
print(f"VAL MIX (picks best epoch): {len(own_val)} own + {len(iam_val)} IAM + {len(other_val)} collected others = {len(own_val)+len(iam_val)+len(other_val)}")
if not (iam_val or other_val):
    print("WARNING: the checkpoint will be picked on your own handwriting only. Add IAM or other-writer pages, or this repeats the v6 problem.")
if share_own > 0.7:
    print("WARNING: your handwriting is still most of the training mix because there are not enough IAM or other-writer lines. Add more of them.")

## 8. Download the PP-OCRv6 medium pretrained weights

In [ ]:
!mkdir -p {PRETRAIN_DIR}
!wget -q -O {PRETRAIN_DIR}/PP-OCRv6_medium_rec_pretrained.pdparams https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_medium_rec_pretrained.pdparams
!ls -la {PRETRAIN_DIR}

## 9. Patch PaddleOCR (required)

Both patches only change *training* behaviour. Evaluation and the exported inference model are unaffected.
Each cell stops with an error if PaddleOCR's code has changed and the patch no longer matches. If that happens, paste the error back to Claude.

In [ ]:
# Patch 1: keep BatchNorm layers in eval mode during training (use stored stats, do not update them)
p = PADDLEOCR_DIR + "/tools/program.py"
s = open(p).read()
if "use stored stats, don't update them" in s:
    print("patch 1 already applied")
else:
    old = "            model.train()\n            profiler.add_profiler_step(profiler_options)"
    new = ("            model.train()\n"
           "            for _m in model.sublayers():\n"
           "                if type(_m).__name__.startswith('BatchNorm'):\n"
           "                    _m.eval()  # use stored stats, don't update them\n"
           "            profiler.add_profiler_step(profiler_options)")
    assert s.count(old) == 1, "patch 1 target not found"
    open(p, "w").write(s.replace(old, new))
    print("patch 1 applied")

# Patch 2: v6 backbone pools to a fixed 40 columns in train mode but keeps the width in eval mode.
# Lines here are longer than 40 characters, so use the width-preserving pooling in train mode too.
p = PADDLEOCR_DIR + "/ppocr/modeling/backbones/rec_lcnetv4.py"
s = open(p).read()
if "keep width in train mode too" in s:
    print("patch 2 already applied")
else:
    old = "            if self.training:\n                x = F.adaptive_avg_pool2d(x, [1, 40])\n            else:"
    new = "            if False:  # keep width in train mode too (lines are longer than 40 chars)\n                x = F.adaptive_avg_pool2d(x, [1, 40])\n            else:"
    assert s.count(old) == 1, "patch 2 target not found"
    open(p, "w").write(s.replace(old, new))
    print("patch 2 applied")

## 10. Write the training config

In [ ]:
def write_config(path, train_list, eval_list, out_dir=None, epochs=None):
    out_dir = out_dir or OUTPUT_DIR
    epochs = epochs or EPOCHS
    config_yaml = f'''
Global:
  model_name: PP-OCRv6_medium_rec
  use_gpu: true
  epoch_num: {epochs}
  log_smooth_window: 20
  print_batch_step: 20
  save_model_dir: {out_dir}
  save_epoch_step: 100
  eval_batch_step: [0, {steps_per_epoch}]
  cal_metric_during_train: true
  pretrained_model: {PRETRAINED}
  checkpoints:
  save_inference_dir:
  use_visualdl: false
  character_dict_path: ppocr/utils/dict/ppocrv6_dict.txt
  max_text_length: &max_text_length 100
  infer_mode: false
  use_space_char: true
  distributed: false

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: {LR_STR}
    warmup_epoch: 2
  regularizer:
    name: L2
    factor: 3.0e-05

Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform:
  Backbone:
    name: PPLCNetV4
    model_size: medium
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: lightsvtr
            dims: 192
            depth: 2
            mlp_ratio: 4.0
            local_kernel: 7
            use_guide: false
          Head:
            fc_decay: 0.00001
      - NRTRHead:
          nrtr_dim: 512
          max_text_length: *max_text_length

Loss:
  name: MultiLoss
  weight_2: 0.0
  loss_config_list:
    - CTCLoss:
    - NRTRLoss:

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: norm_edit_dis

Train:
  dataset:
    name: SimpleDataSet
    data_dir: {DATA_DIR}
    label_file_list: ["{train_list}"]
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - MultiLabelEncode:
          gtc_encode: NRTRLabelEncode
      - RecResizeImg:
          image_shape: [3, 48, {IMG_W}]
      - KeepKeys:
          keep_keys: ["image", "label_ctc", "label_gtc", "length", "valid_ratio"]
  loader:
    shuffle: true
    batch_size_per_card: {BATCH}
    drop_last: true
    num_workers: 2

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: {DATA_DIR}
    label_file_list: ["{eval_list}"]
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - MultiLabelEncode:
          gtc_encode: NRTRLabelEncode
      - RecResizeImg:
          image_shape: [3, 48, {IMG_W}]
      - KeepKeys:
          keep_keys: ["image", "label_ctc", "label_gtc", "length", "valid_ratio"]
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: {BATCH}
    num_workers: 2
'''
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, "w").write(config_yaml)

# training config: validates on the mixed set, which decides which epoch is kept as best_accuracy
write_config(CONFIG_PATH, TRAIN_MIX, VAL_MIX)
print("Config written:", CONFIG_PATH)
print([l.strip() for l in open(CONFIG_PATH).read().splitlines() if any(k in l for k in ["learning_rate", "weight_2", "epoch_num", "save_model_dir"])])

## 11. Baseline: score the untrained (stock) v6 weights on every data source

`eval_list()` scores one checkpoint on one list. `report()` prints a table across all sources. Stock is expected at about 0.82-0.87 on your own lines.
Write down the stock numbers for the IAM and other-writer columns, because those are the ones the new model has to beat.

In [ ]:
import subprocess, re

def eval_list(list_path, ckpt):
    cfg = WORK + "/eval_tmp.yml"
    write_config(cfg, TRAIN_MIX, list_path)
    r = subprocess.run(["python", "tools/eval.py", "-c", cfg, "-o", f"Global.pretrained_model={ckpt}"],
                       capture_output=True, text=True, cwd=PADDLEOCR_DIR)
    out = r.stdout + r.stderr
    m = re.findall(r"norm_edit_dis:\s*([\d.]+)", out)
    if not m:
        print(out[-800:])
    return float(m[-1]) if m else float("nan")

def report(models):
    sets = {"own val": VAL_OWN, "own test": TEST_OWN}
    if iam_val:   sets["IAM val"] = VAL_IAM
    if other_val: sets["others val"] = VAL_OTHER
    print(f"{'model':14}" + "".join(f"{k:>12}" for k in sets) + "   (norm_edit_dis, higher is better)")
    for name, ckpt in models.items():
        print(f"{name:14}" + "".join(f"{eval_list(v, ckpt):12.4f}" for v in sets.values()))

report({"stock": PRETRAINED})

## 12. Train

Watch the `cur metric ... norm_edit_dis` line after each epoch. That number is on the mixed validation set, so it moves less than your old own-hand-only number did.
`best_accuracy` is saved whenever it improves. If it stops improving for 3-4 epochs you can stop the cell (Runtime > Interrupt); `best_accuracy` is already saved.

In [ ]:
%cd /content/PaddleOCR
!python tools/train.py -c {CONFIG_PATH}

## 13. Final comparison

The new model is worth shipping only if it beats stock on **IAM val and others val** (other writers) and is not clearly worse than stock on your own val and test.
`own test` was never used to pick the checkpoint, so it is the honest number for your own handwriting.

In [ ]:
report({"stock": PRETRAINED, "v7 (best)": OUTPUT_DIR + "/best_accuracy"})

## 14. Back up the model (do this right away)

Downloads just `best_accuracy.pdparams` (about 230 MB). That single file is enough to resume or re-export later: upload it to `/content/output/journal_v7/` in a fresh session.

In [ ]:
from google.colab import files
files.download(OUTPUT_DIR + "/best_accuracy.pdparams")

## 15. Export the inference model for your local script

In [ ]:
%cd /content/PaddleOCR
!python tools/export_model.py -c {CONFIG_PATH} -o Global.pretrained_model={OUTPUT_DIR}/best_accuracy Global.save_inference_dir=/content/output/journal_v7_infer
!ls -la /content/output/journal_v7_infer
!cd /content/output && zip -qr /content/journal_v7_infer.zip journal_v7_infer
from google.colab import files
files.download("/content/journal_v7_infer.zip")

## 16. Next steps (on your computer)

1. Unzip the download so the repo contains `journal_v7_infer/` (next to `journal_v6_infer/`).
2. Score it on pages the model never saw, against stock and v6:
   ```
   python ocr_training/evaluate_ocr.py eval_others/ --rec-model-dir journal_v7_infer --compare-stock
   python ocr_training/evaluate_ocr.py eval_others/ --rec-model-dir journal_v6_infer
   python ocr_training/evaluate_ocr.py eval_set/ --rec-model-dir journal_v7_infer --compare-stock
   ```
3. Make `journal_v7_infer` your default only if it beats stock on `eval_others/` and is not worse than stock on `eval_set/` (your own handwriting).
4. If it still loses on other writers, the fix is more writers in `other_writers/` (see the data collection plan), not more epochs.
5. If a patch cell fails with "patch target not found", PaddleOCR's code changed since this notebook was written. Paste the error to Claude.